## Silver to Gold: Building BI Ready Tables


In [1]:
import duckdb
import pandas as pd

In [2]:
# Connect database
try:
    con = duckdb.connect("../data/finance_analytics_light_raw.duckdb")
    print("✅Successfully connected to the database")
except ValueError as e:
    print(f"Failed connected to the database{e}")

✅Successfully connected to the database


#### Show Schema


In [3]:
# Show All Schema
con.sql(
    """
    SHOW SCHEMAS
    """
)

┌─────────────────────────────┬─────────────┬─────────┐
│        database_name        │ schema_name │ current │
│           varchar           │   varchar   │ boolean │
├─────────────────────────────┼─────────────┼─────────┤
│ finance_analytics_light_raw │ bronze      │ false   │
│ finance_analytics_light_raw │ gold        │ false   │
│ finance_analytics_light_raw │ main        │ true    │
│ finance_analytics_light_raw │ silver      │ false   │
└─────────────────────────────┴─────────────┴─────────┘

In [4]:
# Show all tables from silver schema
con.sql(
    """
    SHOW TABLES FROM finance_analytics_light_raw.silver
    """
)

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ account_mapping │
│ accounts        │
│ gl_transactions │
│ stores          │
└─────────────────┘

#### DimAccounts


In [5]:
# Check silver accounts table
con.sql(
    """
    SELECT
        *
    FROM silver.accounts
    """
)

┌────────────────┬────────────────────┬───────────────────┬──────────┐
│ account_number │    account_name    │   account_type    │ currency │
│     int32      │      varchar       │      varchar      │ varchar  │
├────────────────┼────────────────────┼───────────────────┼──────────┤
│           3000 │ Product Sales      │ REVENUE           │ NOK      │
│           4000 │ Cost of Goods Sold │ COGS              │ NOK      │
│           5000 │ Salaries & Wages   │ OPERATING EXPENSE │ NOK      │
│           5100 │ Rent Expense       │ OPERATING EXPENSE │ NOK      │
│           5200 │ Marketing Expense  │ OPERATING EXPENSE │ NOK      │
│           7000 │ Interest Expense   │ FINANCIAL         │ NOK      │
└────────────────┴────────────────────┴───────────────────┴──────────┘

In [ ]:
# Create gold dimensional accounts table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE gold.dim_accounts AS 
        SELECT
            account_number  AS AccountNumber,
            account_name    AS AccountName,
            account_type    AS AccountType,
            currency        AS AccountCurrency
        FROM silver.accounts
        ORDER BY AccountNumber
        """
    )
    print("✅ Created Dimensional Accounts")
except ValueError as error:
    print(f"Failed create gold dimensional accounts table, {error}")

✅ Created Dimensional Accounts


In [ ]:
# Check gold dimensional accounts
con.sql(
    """
    SELECT
        *
    FROM gold.dim_accounts
    """
)

┌───────────────┬────────────────────┬───────────────────┬─────────────────┐
│ AccountNumber │    AccountName     │    AccountType    │ AccountCurrency │
│     int32     │      varchar       │      varchar      │     varchar     │
├───────────────┼────────────────────┼───────────────────┼─────────────────┤
│          3000 │ Product Sales      │ REVENUE           │ NOK             │
│          4000 │ Cost of Goods Sold │ COGS              │ NOK             │
│          5000 │ Salaries & Wages   │ OPERATING EXPENSE │ NOK             │
│          5100 │ Rent Expense       │ OPERATING EXPENSE │ NOK             │
│          5200 │ Marketing Expense  │ OPERATING EXPENSE │ NOK             │
│          7000 │ Interest Expense   │ FINANCIAL         │ NOK             │
└───────────────┴────────────────────┴───────────────────┴─────────────────┘

#### DimStores


In [8]:
# Check silver stores table
con.sql(
    """
    SELECT
        *
    FROM silver.stores
    """
)

┌────────────┬───────────────────┬────────────┬─────────┬──────────┐
│ store_code │    store_name     │ store_type │ country │  region  │
│  varchar   │      varchar      │  varchar   │ varchar │ varchar  │
├────────────┼───────────────────┼────────────┼─────────┼──────────┤
│ NO001      │ NRG Oslo City     │ STORE      │ Norway  │ East     │
│ NO002      │ NRG Bergen        │ STORE      │ Norway  │ West     │
│ NO003      │ NRG Trondheim     │ STORE      │ Norway  │ Mid      │
│ SE001      │ NRG Stockholm     │ STORE      │ Sweden  │ East     │
│ SE002      │ NRG Gothenburg    │ STORE      │ Sweden  │ West     │
│ EC_NO      │ NRG Online Norway │ ECOM       │ Norway  │ National │
│ EC_SE      │ NRG Online Sweden │ ECOM       │ Sweden  │ National │
└────────────┴───────────────────┴────────────┴─────────┴──────────┘

In [10]:
# Create gold dimensional stores table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE gold.dim_stores AS 
        SELECT
            store_code              AS StoreCode,
            TRIM(store_name)        AS StoreName,
            UPPER(TRIM(store_type)) AS StoreType,
            country                 AS Country,
            region                  AS Region
        FROM silver.stores
        ORDER BY StoreCode
        """
    )
    print("✅ Created dimensional stores table")
except ValueError as error:
    print(f"Failed create gold dimensional stores table, {error}")

✅ Created dimensional stores table


In [11]:
# Check dimensional stores table
con.sql(
    """
    SELECT
        *
    FROM gold.dim_stores
    """
)

┌───────────┬───────────────────┬───────────┬─────────┬──────────┐
│ StoreCode │     StoreName     │ StoreType │ Country │  Region  │
│  varchar  │      varchar      │  varchar  │ varchar │ varchar  │
├───────────┼───────────────────┼───────────┼─────────┼──────────┤
│ EC_NO     │ NRG Online Norway │ ECOM      │ Norway  │ National │
│ EC_SE     │ NRG Online Sweden │ ECOM      │ Sweden  │ National │
│ NO001     │ NRG Oslo City     │ STORE     │ Norway  │ East     │
│ NO002     │ NRG Bergen        │ STORE     │ Norway  │ West     │
│ NO003     │ NRG Trondheim     │ STORE     │ Norway  │ Mid      │
│ SE001     │ NRG Stockholm     │ STORE     │ Sweden  │ East     │
│ SE002     │ NRG Gothenburg    │ STORE     │ Sweden  │ West     │
└───────────┴───────────────────┴───────────┴─────────┴──────────┘

#### DimAccountMapping


In [12]:
# Check silver accounts mapping
con.sql(
    """
    SELECT
        *
    FROM silver.account_mapping    
    """
)

┌────────────────┬────────────────────┬────────────────────┬────────────────┬────────────┬───────────────────────────────────────────────────────────────────┐
│ account_number │    account_name    │      pl_line       │ statement_type │ sort_order │                               notes                               │
│     int32      │      varchar       │      varchar       │    varchar     │   int32    │                              varchar                              │
├────────────────┼────────────────────┼────────────────────┼────────────────┼────────────┼───────────────────────────────────────────────────────────────────┤
│           3000 │ Product Sales      │ Revenue            │ P&L            │         10 │ NULL                                                              │
│           3010 │ Online Sales       │ Revenue            │ P&L            │         10 │ New online channel – mapping not yet confirmed                    │
│           4000 │ Cost of Goods Sold │ COGS  

In [14]:
# Create dimensional account mapping
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE gold.dim_account_mapping AS
        SELECT 
            account_number  AS AccountNumber,
            account_name    AS AccountName,
            pl_line         AS PLLine,
            statement_type  AS StatementType,
            sort_order      AS SortOrder,
            notes           AS Notes
        FROM silver.account_mapping
        ORDER BY SortOrder, AccountNumber
        """
    )
    print("✅ Created gold dimensional account mapping table")
except ValueError as error:
    print(f"Failed create gold dimensional account mapping table, {error}")

✅ Created gold dimensional account mapping table


In [ ]:
# Check dimensional account mapping table
con.sql(
    """
    SELECT
    *
    FROM gold.dim_account_mapping
    """
)

┌───────────────┬────────────────────┬────────────────────┬───────────────┬───────────┬───────────────────────────────────────────────────────────────────┐
│ AccountNumber │    AccountName     │       PLLine       │ StatementType │ SortOrder │                               Notes                               │
│     int32     │      varchar       │      varchar       │    varchar    │   int32   │                              varchar                              │
├───────────────┼────────────────────┼────────────────────┼───────────────┼───────────┼───────────────────────────────────────────────────────────────────┤
│          3000 │ Product Sales      │ Revenue            │ P&L           │        10 │ NULL                                                              │
│          3010 │ Online Sales       │ Revenue            │ P&L           │        10 │ New online channel – mapping not yet confirmed                    │
│          4000 │ Cost of Goods Sold │ COGS               │ P&L 

#### FactGLTransactions


In [ ]:
# Check silver gl transactions table
con.sql(
    """
    SELECT
    *
    FROM silver.gl_transactions
    """
)

┌────────────────┬──────────────────┬────────────┬────────────────┬──────────────┬─────────┬──────────┬─────────────────┬──────────────────────────────────────────────┐
│ transaction_id │ transaction_date │ store_code │ account_number │ amount_local │ dc_flag │ currency │ document_number │                 description                  │
│     int64      │       date       │  varchar   │     int32      │    double    │ varchar │ varchar  │     varchar     │                   varchar                    │
├────────────────┼──────────────────┼────────────┼────────────────┼──────────────┼─────────┼──────────┼─────────────────┼──────────────────────────────────────────────┤
│              1 │ 2025-03-12       │ SE001      │           4000 │     -7149.34 │ DEBIT   │ NOK      │ DOC-2025-000001 │ Cost of Goods Sold - Recently thought month. │
│              2 │ 2025-09-24       │ SE001      │           4000 │     -6023.02 │ DEBIT   │ NOK      │ DOC-2025-000002 │ Cost of Goods Sold - Per send.   

In [17]:
# Create fact gl transactions table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE gold.fact_gl_transactions AS
        SELECT
            transaction_id      AS TransactionID,
            transaction_date    AS TransactionDate,
            store_code          AS StoreCode,
            account_number      AS AccountNumber,
            dc_flag             AS DCFlag,
            amount_local        AS AmountLocal,
            currency            AS Currency,
            document_number     AS DocumentNumber,
            description         AS Description
        FROM silver.gl_transactions
        ORDER BY TransactionID
        """
    )
    print("✅ Created gold fact gl transactions table")
except ValueError as error:
    print(f"Failed create gold fact gl transactions table, {error}")

✅ Created gold fact gl transactions table


In [18]:
# Check fact gl transaction table
con.sql(
    """
    SELECT
    *
    FROM gold.fact_gl_transactions
    """
)

┌───────────────┬─────────────────┬───────────┬───────────────┬─────────┬─────────────┬──────────┬─────────────────┬──────────────────────────────────────────────┐
│ TransactionID │ TransactionDate │ StoreCode │ AccountNumber │ DCFlag  │ AmountLocal │ Currency │ DocumentNumber  │                 Description                  │
│     int64     │      date       │  varchar  │     int32     │ varchar │   double    │ varchar  │     varchar     │                   varchar                    │
├───────────────┼─────────────────┼───────────┼───────────────┼─────────┼─────────────┼──────────┼─────────────────┼──────────────────────────────────────────────┤
│             1 │ 2025-03-12      │ SE001     │          4000 │ DEBIT   │    -7149.34 │ NOK      │ DOC-2025-000001 │ Cost of Goods Sold - Recently thought month. │
│             2 │ 2025-09-24      │ SE001     │          4000 │ DEBIT   │    -6023.02 │ NOK      │ DOC-2025-000002 │ Cost of Goods Sold - Per send.               │
│             3 

#### FactGLTransactionsMonthly


In [19]:
# Create gold aggregate fact gl transactions table monthly
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE gold.fact_gl_transactions_monthly AS
        WITH base AS (
            SELECT 
                TransactionDate ,
                StoreCode,
                AccountNumber,
                AmountLocal
            FROM gold.fact_gl_transactions
        )
        SELECT 
            DATE_TRUNC('month', TransactionDate) AS TransactionMonth,
            StoreCode,
            AccountNumber,
            SUM(AmountLocal)            AS AmountLocal
        FROM base
        GROUP BY 
            TransactionMonth,
            StoreCode,
            AccountNumber
        ORDER BY 
            TransactionMonth,
            StoreCode,
            AccountNumber
        """
    )
    print("✅ Created gold fact gl transactions monthly table")
except ValueError as error:
    print(f"Failed create gold fact gl transactions monthly table, {error}")

✅ Created gold fact gl transactions monthly table


In [20]:
con.sql(
    """
    SELECT
        *
    FROM gold.fact_gl_transactions_monthly
    """
)

┌─────────────────────┬───────────┬───────────────┬─────────────────────┐
│  TransactionMonth   │ StoreCode │ AccountNumber │     AmountLocal     │
│      timestamp      │  varchar  │     int32     │       double        │
├─────────────────────┼───────────┼───────────────┼─────────────────────┤
│ 2025-01-01 00:00:00 │ EC_NO     │          3000 │  128416.98999999999 │
│ 2025-01-01 00:00:00 │ EC_NO     │          4000 │          -132913.18 │
│ 2025-01-01 00:00:00 │ EC_NO     │          5000 │ -112326.34999999999 │
│ 2025-01-01 00:00:00 │ EC_NO     │          5100 │  -75480.20999999999 │
│ 2025-01-01 00:00:00 │ EC_NO     │          5200 │ -45768.810000000005 │
│ 2025-01-01 00:00:00 │ EC_NO     │          7000 │           -68296.07 │
│ 2025-01-01 00:00:00 │ EC_SE     │          3000 │  121805.41000000002 │
│ 2025-01-01 00:00:00 │ EC_SE     │          4000 │ -118762.15999999999 │
│ 2025-01-01 00:00:00 │ EC_SE     │          5000 │            -83121.2 │
│ 2025-01-01 00:00:00 │ EC_SE     │   

#### FactScenarioGLTransactions


In [21]:
# Scenario if get a GL increase and decrease
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE gold.fact_scenario_gl_transactions AS
        
        -- Aggregate transactions to month + store + account
        WITH base AS (
            SELECT
                DATE_TRUNC('month', TransactionDate) AS TransactionMonth,
                StoreCode,
                AccountNumber,
                SUM(AmountLocal)                                                    AS AmountLocal
            FROM gold.fact_gl_transactions
            GROUP BY
                TransactionMonth,
                StoreCode,
                AccountNumber
        )

        -- Actual values
        SELECT
            TransactionMonth,
            StoreCode,
            AccountNumber,
            'Actual'                                                                AS Scenario,
            AmountLocal                                                             AS AmountLocal
        FROM base

        UNION ALL

        -- Best-case scenario (revenue up, costs down)
        SELECT
            TransactionMonth,
            StoreCode,
            AccountNumber,
            'BestCase'                                                              AS Scenario,
            CASE
                WHEN AmountLocal >= 0 THEN AmountLocal * 1.10
                ELSE AmountLocal * 0.90
            END                                                                     AS AmountLocal
        FROM base

        UNION ALL

        -- Worst-case scenario (revenue down, costs up)
        SELECT
            TransactionMonth,
            StoreCode,
            AccountNumber,
            'WorstCase'                                                             AS Scenario,
            CASE
                WHEN AmountLocal >= 0 THEN AmountLocal * 0.90
                ELSE AmountLocal * 1.10
            END                                                                     AS AmountLocal
        FROM base

        ORDER BY
            TransactionMonth,
            StoreCode,
            AccountNumber,
            Scenario;
        """
    )
    print("✅ Created gold fact scenario gl transactions table")
except ValueError as error:
    print(f"Failed create gold fact scenario gl transactions table, {error}")

✅ Created gold fact scenario gl transactions table


In [ ]:
# Check gold fact scenario gl transactions table
con.sql(
    """
    SELECT
    *
    FROM gold.fact_scenario_gl_transactions
    """
)

┌─────────────────────┬───────────┬───────────────┬───────────┬─────────────────────┐
│  TransactionMonth   │ StoreCode │ AccountNumber │ Scenario  │     AmountLocal     │
│      timestamp      │  varchar  │     int32     │  varchar  │       double        │
├─────────────────────┼───────────┼───────────────┼───────────┼─────────────────────┤
│ 2025-01-01 00:00:00 │ EC_NO     │          3000 │ Actual    │  128416.98999999999 │
│ 2025-01-01 00:00:00 │ EC_NO     │          3000 │ BestCase  │          141258.689 │
│ 2025-01-01 00:00:00 │ EC_NO     │          3000 │ WorstCase │          115575.291 │
│ 2025-01-01 00:00:00 │ EC_NO     │          4000 │ Actual    │          -132913.18 │
│ 2025-01-01 00:00:00 │ EC_NO     │          4000 │ BestCase  │         -119621.862 │
│ 2025-01-01 00:00:00 │ EC_NO     │          4000 │ WorstCase │         -146204.498 │
│ 2025-01-01 00:00:00 │ EC_NO     │          5000 │ Actual    │ -112326.34999999999 │
│ 2025-01-01 00:00:00 │ EC_NO     │          5000 │ Be

In [23]:
con.close()